# 🚢 Titanic — Modelo 2: Random Forest

Este notebook treina e avalia um classificador Random Forest com GridSearchCV.

**Pipeline:**
```
ColumnDropper → CategorizerFeatures → OrdinalEncoder → OneHotEncoder → Imputer → Scaler → RandomForestClassifier
```

---
> **Fluxo:** `01_EDA` → `02_Logistic_Regression` → **`03_Random_Forest`** → `04_Voting_Classifier`

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from data_transform import build_preprocessing_pipeline

## 2. Carregamento dos dados

In [ ]:
df_train  = pd.read_csv('dataset/titanicc/train.csv')
df_test   = pd.read_csv('dataset/titanicc/test.csv')
df_labels = pd.read_csv('dataset/titanicc/gender_submission.csv')

FEATURES = ['PassengerId', 'Pclass', 'Name', 'Sex', 'Age',
            'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

X_train = df_train[FEATURES]
y_train = df_train['Survived']

X_test  = df_test
y_test  = df_labels['Survived'].to_numpy()

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")

## 3. Pré-processamento + GridSearchCV

In [ ]:
ppl_transform = build_preprocessing_pipeline()
X_train_ppl   = ppl_transform.fit_transform(X_train)

param_grid = {
    'n_estimators'    : [25, 50, 75, 100],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
}

grid_search = GridSearchCV(
    RandomForestClassifier(bootstrap=True, max_features='sqrt', random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_ppl, y_train)

best_params = grid_search.best_params_
print("Melhores parâmetros :", best_params)
print(f"Melhor score (CV=5) : {grid_search.best_score_:.4f}")

## 4. Treino e avaliação no conjunto de teste

In [ ]:
model = make_pipeline(
    build_preprocessing_pipeline(),
    RandomForestClassifier(
        bootstrap=True, max_depth=None,
        max_features='sqrt', random_state=42,
        **best_params
    )
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Morreu', 'Sobreviveu']))

## 5. Matriz de Confusão

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Morreu', 'Sobreviveu'],
    cmap='Blues', ax=ax
)
ax.set_title('Random Forest — Matriz de Confusão', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Importância das Features

In [ ]:
ppl   = build_preprocessing_pipeline()
X_tr  = ppl.fit_transform(X_train)

rf_final = RandomForestClassifier(
    bootstrap=True, max_depth=None,
    max_features='sqrt', random_state=42,
    **best_params
)
rf_final.fit(X_tr, y_train)

importances = pd.Series(rf_final.feature_importances_).sort_values(ascending=False).head(15)
importances.plot(kind='barh', figsize=(9, 6), color='steelblue', edgecolor='white')
plt.gca().invert_yaxis()
plt.title('Top 15 Features — Random Forest', fontsize=13)
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

---
**Próximo passo:** `04_Voting_Classifier.ipynb`